In [ ]:
MAPPING FILE: mplt_CDM_ROW_WID.txt
====================================================================================================

"""
ETL Pipeline: mplt_CDM_ROW_WID
Migrated from IICS mapping: mplt_CDM_ROW_WID

Purpose:
This pipeline retrieves the maximum ROW_WID and associated TABLE_NAME from a target table using a lookup transformation,
calculates a new ROW_WID using conditional logic, and inserts the calculated ROW_WID into the target table.

"""

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import logging
import os

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Utility functions
def read_table(table_name: str, local_file_path: str = None) -> DataFrame:
    """
    Read data from catalog (Databricks) or local file based on environment.

    Args:
        table_name: Full table name in format "catalog.database.table" (e.g., "schema_cdm.target_table_name").
        local_file_path: Local CSV file path for non-Databricks environment.

    Returns:
        DataFrame from catalog or local file.
    """
    is_databricks = 'DATABRICKS_RUNTIME_VERSION' in os.environ

    if is_databricks:
        logger.info(f"Reading from catalog: {table_name}")
        return spark.table(table_name)
    else:
        if local_file_path is None:
            raise Exception(f"Local file path required for {table_name}")
        logger.info(f"Reading from local file: {local_file_path}")
        return spark.read.csv(local_file_path, header=True, inferSchema=True)

def log_df_info(df: DataFrame, step_name: str):
    """
    Log DataFrame information for debugging.

    Args:
        df: DataFrame to log.
        step_name: Description of the step.
    """
    try:
        logger.info(f"{step_name}: {len(df.columns)} columns")
        logger.info(f"Schema: {[f.name + ':' + str(f.dataType) for f in df.schema.fields]}")
    except Exception as e:
        logger.error(f"Error logging DataFrame info: {str(e)}")


class CDMRowWidPipeline:
    """
    ETL Pipeline for calculating and inserting ROW_WID.

    Sources: Custom Table
    Targets: Output Table
    Transformation Logic:
    - Lookup maximum ROW_WID and associated TABLE_NAME.
    - Calculate new ROW_WID using conditional logic.
    - Insert calculated ROW_WID into the target table.
    """

    def __init__(self, spark: SparkSession, config: dict):
        self.spark = spark
        self.config = config
        self._configure_spark()

    def _configure_spark(self):
        """
        Set Spark configurations for optimal performance.
        """
        configs = {
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.shuffle.partitions": str(self.config.get('shuffle_partitions', 200))
        }

        for key, value in configs.items():
            self.spark.conf.set(key, value)

    def lookup_max_row_wid(self, in_table_name: str) -> DataFrame:
        """
        Perform lookup to retrieve maximum ROW_WID and associated TABLE_NAME.

        Args:
            in_table_name: Input table name for lookup.

        Returns:
            DataFrame with ROW_WID and TABLE_NAME.
        """
        logger.info("Performing lookup for maximum ROW_WID")

        lookup_table_name = self.config.get('lookup_table', 'schema_cdm.target_table_name')
        lookup_local_path = self.config.get('lookup_local_path', None)

        lookup_df = read_table(
            table_name=lookup_table_name,
            local_file_path=lookup_local_path
        )

        # Filter for the specific TABLE_NAME and retrieve ROW_WID and TABLE_NAME
        lookup_result = lookup_df.filter(F.col("TABLE_NAME") == in_table_name).select(
            F.coalesce(F.max(F.col("ROW_WID")), F.lit(0)).alias("ROW_WID"),
            F.lit(in_table_name).alias("TABLE_NAME")
        )

        log_df_info(lookup_result, "Lookup Result")
        return lookup_result

    def calculate_new_row_wid(self, lookup_df: DataFrame) -> DataFrame:
        """
        Calculate new ROW_WID using lookup results and conditional logic.

        Args:
            lookup_df: DataFrame containing lookup results.

        Returns:
            DataFrame with calculated ROW_WID.
        """
        logger.info("Calculating new ROW_WID")

        # Add conditional logic and increment ROW_WID
        calculated_df = lookup_df.withColumn(
            "ROW_WID",
            F.col("ROW_WID") + F.lit(1)
        )

        log_df_info(calculated_df, "Calculated ROW_WID")
        return calculated_df

    def insert_row_wid(self, calculated_df: DataFrame):
        """
        Insert calculated ROW_WID into the target table.

        Args:
            calculated_df: DataFrame containing calculated ROW_WID.
        """
        logger.info("Inserting ROW_WID into target table")

        target_path = self.config.get('target_path', '/path/to/target')

        log_df_info(calculated_df, "Before Load")

        # Write to Delta table
        (calculated_df.write
            .format("delta")
            .mode(self.config.get('write_mode', 'append'))
            .save(target_path)
        )

        # Optimize target table
        if self.config.get('optimize_target', True):
            logger.info("Optimizing target table")
            zorder_cols = self.config.get('zorder_columns', [])
            if zorder_cols:
                self.spark.sql(f"""
                    OPTIMIZE delta.`{target_path}`
                    ZORDER BY ({', '.join(zorder_cols)})
                """)

        logger.info("Data load complete")

    def execute(self):
        """
        Execute the complete ETL pipeline.
        """
        logger.info("Starting ETL pipeline: CDMRowWidPipeline")

        try:
            # Step 1: Lookup Maximum ROW_WID
            in_table_name = self.config.get('in_table_name', 'default_table_name')
            lookup_df = self.lookup_max_row_wid(in_table_name)

            # Step 2: Calculate New ROW_WID
            calculated_df = self.calculate_new_row_wid(lookup_df)

            # Step 3: Insert ROW_WID into Target Table
            self.insert_row_wid(calculated_df)

            logger.info("ETL pipeline completed successfully")

        except Exception as e:
            logger.error(f"ETL pipeline failed: {str(e)}")
            raise


# Configuration
config = {
    # Lookup configuration
    'lookup_table': 'schema_cdm.target_table_name',
    'lookup_local_path': r'path/to/lookup.csv',

    # Target configuration
    'target_path': '/path/to/target',
    'write_mode': 'append',
    'zorder_columns': ['ROW_WID'],

    # Performance configuration
    'shuffle_partitions': 200,
    'optimize_target': True,

    # Input configuration
    'in_table_name': 'CUSTOM_TABLE'
}

# Execute
if __name__ == "__main__":
    spark = SparkSession.builder.appName("CDM_ROW_WID_Pipeline").getOrCreate()
    pipeline = CDMRowWidPipeline(spark, config)
    pipeline.execute()
MAPPING FILE: mplt_CDM_BATCH_ID.txt
====================================================================================================

"""
ETL Pipeline: mplt_CDM_BATCH_ID
Migrated from IICS mapping: mplt_CDM_BATCH_ID

Description:
This pipeline processes batch IDs by checking for null values, performing a lookup to retrieve the maximum batch ID for a given source name, and outputs either the lookup value or a default value (-999) for null cases.
"""

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import logging
import os

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Utility functions
def read_table(table_name: str, local_file_path: str = None) -> DataFrame:
    """
    Read data from catalog (Databricks) or local file based on environment.

    Args:
        table_name: Full table name in format "catalog.database.table" (e.g., "CDM.CDM_BATCH_CTRLID").
        local_file_path: Local CSV file path for non-Databricks environment.

    Returns:
        DataFrame from catalog or local file.
    """
    is_databricks = 'DATABRICKS_RUNTIME_VERSION' in os.environ

    if is_databricks:
        logger.info(f"Reading from catalog: {table_name}")
        return spark.table(table_name)
    else:
        if local_file_path is None:
            raise Exception(f"Local file path required for {table_name}")
        logger.info(f"Reading from local file: {local_file_path}")
        return spark.read.csv(local_file_path, header=True, inferSchema=True)

def log_df_info(df: DataFrame, step_name: str):
    """
    Log DataFrame information for debugging.

    Args:
        df: DataFrame to log.
        step_name: Description of the step.
    """
    try:
        logger.info(f"{step_name}: {len(df.columns)} columns")
        logger.info(f"Schema: {[f.name + ':' + str(f.dataType) for f in df.schema.fields]}")
    except Exception as e:
        logger.error(f"Error logging DataFrame info: {str(e)}")

class CDMBatchIDPipeline:
    """
    ETL Pipeline for processing batch IDs.

    Sources: CDM.CDM_BATCH_CTRLID
    Targets: o_BATCH_ID
    Transformation Logic: Null handling, lookup enrichment, and default value assignment.
    """

    def __init__(self, spark: SparkSession, config: dict):
        self.spark = spark
        self.config = config
        self._configure_spark()

    def _configure_spark(self):
        """
        Set Spark configurations for optimal performance.
        """
        configs = {
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.shuffle.partitions": str(self.config.get('shuffle_partitions', 200))
        }

        for key, value in configs.items():
            self.spark.conf.set(key, value)

    def extract(self) -> DataFrame:
        """
        Extract data from source.

        Optimizations:
        - Predicate pushdown for filters.
        - Column pruning to select only required fields.
        """
        logger.info("Starting data extraction")

        table_name = self.config.get('source_table', 'CDM.CDM_BATCH_CTRLID')
        local_file_path = self.config.get('source_local_path', None)

        df = read_table(
            table_name=table_name,
            local_file_path=local_file_path
        )

        log_df_info(df, "Source: CDM_BATCH_CTRLID")

        # Apply filters if specified (predicate pushdown)
        if 'filter_conditions' in self.config:
            for condition in self.config['filter_conditions']:
                df = df.filter(condition)

        # Select only required columns (column pruning)
        if 'required_columns' in self.config:
            df = df.select(*self.config['required_columns'])

        logger.info("Data extraction complete")
        return df

    def transform(self, df: DataFrame) -> DataFrame:
        """
        Apply business transformations.

        Transformation Logic:
        - Lookup enrichment.
        - Null handling and default value assignment.
        """
        logger.info("Applying transformations")

        # Perform lookup enrichment
        df = self._perform_lookup(df)

        # Apply null handling and default value assignment
        df = self._apply_null_handling(df)

        log_df_info(df, "After transformations")
        return df

    def _perform_lookup(self, df: DataFrame) -> DataFrame:
        """
        Perform lookup to retrieve maximum batch ID for the given source name.

        Args:
            df: Input DataFrame.

        Returns:
            DataFrame enriched with lookup values.
        """
        logger.info("Performing lookup enrichment")

        lookup_table_name = self.config.get('lookup_table', 'CDM.CDM_BATCH_CTRLID')
        lookup_local_path = self.config.get('lookup_local_path', None)

        lookup_df = read_table(
            table_name=lookup_table_name,
            local_file_path=lookup_local_path
        )

        # Perform lookup using broadcast join for optimization
        enriched_df = df.join(
            F.broadcast(lookup_df.select(
                F.col("SOURCE_NAME").alias("lookup_SOURCE_NAME"),
                F.col("BATCH_ID").alias("lookup_BATCH_ID")
            )),
            df.SOURCE_NAME == lookup_df.lookup_SOURCE_NAME,
            "left"
        ).drop("lookup_SOURCE_NAME")

        return enriched_df

    def _apply_null_handling(self, df: DataFrame) -> DataFrame:
        """
        Apply null handling and default value assignment.

        Args:
            df: Input DataFrame.

        Returns:
            DataFrame with null values replaced by default values.
        """
        logger.info("Applying null handling and default value assignment")

        return df.withColumn(
            "o_BATCH_ID",
            F.when(F.col("lookup_BATCH_ID").isNull(), F.lit(-999)).otherwise(F.col("lookup_BATCH_ID"))
        ).select(
            F.col("o_BATCH_ID"),
            F.col("SOURCE_NAME")
        )

    def load(self, df: DataFrame):
        """
        Load data to target.

        Optimizations:
        - Delta Lake for ACID compliance.
        - Partitioning for query performance.
        """
        logger.info("Starting data load")

        target_path = self.config.get('target_path', '/path/to/target')

        log_df_info(df, "Before load")

        (df.write
            .format("delta")
            .mode(self.config.get('write_mode', 'overwrite'))
            .partitionBy(*self.config.get('partition_columns', []))
            .option("overwriteSchema", "true")
            .save(target_path)
        )

        # Optimize target table
        if self.config.get('optimize_target', True):
            logger.info("Optimizing target table")
            zorder_cols = self.config.get('zorder_columns', [])
            if zorder_cols:
                self.spark.sql(f"""
                    OPTIMIZE delta.`{target_path}`
                    ZORDER BY ({', '.join(zorder_cols)})
                """)

        logger.info("Data load complete")

    def execute(self):
        """
        Execute the complete ETL pipeline.
        """
        logger.info(f"Starting ETL pipeline: {self.__class__.__name__}")

        try:
            # Extract
            source_df = self.extract()

            # Transform
            transformed_df = self.transform(source_df)

            # Load
            self.load(transformed_df)

            logger.info("ETL pipeline completed successfully")

        except Exception as e:
            logger.error(f"ETL pipeline failed: {str(e)}")
            raise


# Configuration
config = {
    # Source configuration
    'source_table': 'CDM.CDM_BATCH_CTRLID',
    'source_local_path': r'path/to/CDM_BATCH_CTRLID.csv',

    # Target configuration
    'target_path': '/path/to/target',
    'write_mode': 'overwrite',
    'partition_columns': ['SOURCE_NAME'],
    'zorder_columns': ['o_BATCH_ID', 'SOURCE_NAME'],

    # Transformation configuration
    'required_columns': ['SOURCE_NAME'],
    'filter_conditions': [],
    'aggregation_required': False,

    # Lookup configuration
    'lookup_table': 'CDM.CDM_BATCH_CTRLID',
    'lookup_local_path': r'path/to/lookup.csv',

    # Performance configuration
    'shuffle_partitions': 200,
    'optimize_target': True
}

# Execute
if __name__ == "__main__":
    spark = SparkSession.builder.appName("CDM_BATCH_ID_Pipeline").getOrCreate()
    pipeline = CDMBatchIDPipeline(spark, config)
    pipeline.execute()
MAPPING FILE: m_CDM_W_CLAIM_CD_SCD3_IU.txt
====================================================================================================
Below is the production-ready PySpark code for the migration of the IICS mapping `m_CDM_W_CLAIM_CD_SCD3_IU` to PySpark. The code is optimized for Spark's distributed computing capabilities and adheres to the provided optimization framework.

---


"""
ETL Pipeline: m_CDM_W_CLAIM_CD_SCD3_IU
Migrated from IICS mapping: m_CDM_W_CLAIM_CD_SCD3_IU
"""

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from typing import Dict
import logging
import os

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Utility functions
def read_table(table_name: str, local_file_path: str = None) -> DataFrame:
    """
    Read data from catalog (Databricks) or local file based on environment.

    Args:
        table_name: Full table name in format "catalog.database.table".
        local_file_path: Local CSV file path for non-Databricks environment.

    Returns:
        DataFrame from catalog or local file.
    """
    is_databricks = 'DATABRICKS_RUNTIME_VERSION' in os.environ

    if is_databricks:
        logger.info(f"Reading from catalog: {table_name}")
        return spark.table(table_name)
    else:
        if local_file_path is None:
            raise Exception(f"Local file path required for {table_name}")
        logger.info(f"Reading from local file: {local_file_path}")
        return spark.read.csv(local_file_path, header=True, inferSchema=True)

def log_df_info(df: DataFrame, step_name: str):
    """
    Log DataFrame information for debugging.

    Args:
        df: DataFrame to log.
        step_name: Description of the step.
    """
    try:
        logger.info(f"{step_name}: {len(df.columns)} columns")
        logger.info(f"Schema: {[f.name + ':' + str(f.dataType) for f in df.schema.fields]}")
    except Exception as e:
        logger.error(f"Error logging DataFrame info: {str(e)}")

class CDMClaimPipeline:
    """
    ETL Pipeline for processing SCD Type 3 logic for BUR field.

    Sources: CDH_GW_BUR
    Targets: W_CLAIM_CD_BUR_SCD3_I (Insert), W_CLAIM_CD_BUR_SCD3_U (Update)
    Transformation Logic: SCD Type 3 handling with flag evaluation and router logic.
    """

    def __init__(self, spark: SparkSession, config: Dict):
        self.spark = spark
        self.config = config
        self._configure_spark()

    def _configure_spark(self):
        """
        Set Spark configurations for optimal performance.
        """
        configs = {
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.shuffle.partitions": str(self.config.get('shuffle_partitions', 200)),
            "spark.sql.autoBroadcastJoinThreshold": "10485760"
        }

        for key, value in configs.items():
            self.spark.conf.set(key, value)

    def extract(self) -> DataFrame:
        """
        Extract data from source table.

        Returns:
            Extracted DataFrame.
        """
        logger.info("Starting data extraction")

        table_name = self.config.get('source_table', 'catalog.database.table')
        local_file_path = self.config.get('source_local_path', None)

        df = read_table(table_name=table_name, local_file_path=local_file_path)

        # Apply SQL override logic
        df = df.select(
            F.col("POLICY_STATE"),
            F.col("BUR"),
            F.lit("GWCDH").alias("SOURCE_NAME")
        )

        log_df_info(df, "Source: CDH_GW_BUR")
        logger.info("Data extraction complete")
        return df

    def transform(self, df: DataFrame) -> DataFrame:
        """
        Apply transformations including expressions, lookups, and flag evaluation.

        Args:
            df: Input DataFrame.

        Returns:
            Transformed DataFrame.
        """
        logger.info("Applying transformations")

        # Expression Transformation
        df = df.select(
            F.col("POLICY_STATE").alias("INTEGRATION_ID"),
            F.col("BUR"),
            F.col("SOURCE_NAME")
        )

        # Lookup Transformation
        lookup_table_name = self.config.get('lookup_table', 'catalog.database.lookup_table')
        lookup_local_path = self.config.get('lookup_local_path', None)

        lookup_df = read_table(table_name=lookup_table_name, local_file_path=lookup_local_path)
        lookup_df = lookup_df.select("LKP_ROW_WID", "LKP_INTEGRATION_ID", "LKP_NEW_BUR")

        df = df.join(
            F.broadcast(lookup_df),
            df.INTEGRATION_ID == lookup_df.LKP_INTEGRATION_ID,
            "left"
        ).select(
            df["*"],
            lookup_df["LKP_ROW_WID"],
            lookup_df["LKP_NEW_BUR"]
        )

        # Flag Evaluation
        df = df.withColumn(
            "o_Flag",
            F.when(F.col("LKP_ROW_WID").isNull(), "I")
            .when(F.md5(F.col("BUR")) == F.md5(F.col("LKP_NEW_BUR")), "NC")
            .otherwise("U")
        ).withColumn(
            "CDM_INSERT_DT",
            F.when(F.col("o_Flag") == "I", F.current_timestamp())
        ).withColumn(
            "CDM_UPDATE_DT",
            F.when(F.col("o_Flag") == "U", F.current_timestamp())
        ).withColumn(
            "TGT_TABLE_NAME",
            F.lit("W_CLAIM_CD_BUR_SCD3")
        )

        log_df_info(df, "After transformations")
        return df

    def load(self, df: DataFrame):
        """
        Load data into target tables.

        Args:
            df: Transformed DataFrame.
        """
        logger.info("Starting data load")

        # Split data into insert and update groups
        insert_df = df.filter(F.col("o_Flag") == "I").select("BUR", "CDM_INSERT_DT", "TGT_TABLE_NAME")
        update_df = df.filter(F.col("o_Flag") == "U").select("BUR", "CDM_UPDATE_DT", "TGT_TABLE_NAME")

        # Load insert data
        insert_target_path = self.config.get('insert_target_path', '/path/to/insert_target')
        (insert_df.write
            .format("delta")
            .mode("append")
            .save(insert_target_path)
        )

        # Load update data
        update_target_path = self.config.get('update_target_path', '/path/to/update_target')
        (update_df.write
            .format("delta")
            .mode("append")
            .save(update_target_path)
        )

        logger.info("Data load complete")

    def execute(self):
        """
        Execute the complete ETL pipeline.
        """
        logger.info(f"Starting ETL pipeline: {self.__class__.__name__}")

        try:
            # Extract
            source_df = self.extract()

            # Transform
            transformed_df = self.transform(source_df)

            # Load
            self.load(transformed_df)

            logger.info("ETL pipeline completed successfully")

        except Exception as e:
            logger.error(f"ETL pipeline failed: {str(e)}")
            raise


# Configuration
config = {
    # Source configuration
    'source_table': 'CDH_GWODS.CDH_GW_BUR',
    'source_local_path': r'path/to/source.csv',

    # Lookup configuration
    'lookup_table': 'CDM.W_CLAIM_CD_BUR_SCD3',
    'lookup_local_path': r'path/to/lookup.csv',

    # Target configuration
    'insert_target_path': '/path/to/insert_target',
    'update_target_path': '/path/to/update_target',

    # Performance configuration
    'shuffle_partitions': 200
}

# Execute
if __name__ == "__main__":
    spark = SparkSession.builder.appName("ETL_Pipeline").getOrCreate()
    pipeline = CDMClaimPipeline(spark, config)
    pipeline.execute()


---

### Key Features:
1. **Optimized Transformations:** Combined multiple transformations into single operations where possible.
2. **Lookup Optimization:** Used broadcast joins for efficient lookup operations.
3. **Router Replacement:** Used filter conditions to split data into insert and update groups.
4. **Error Handling:** Comprehensive logging and exception handling.
5. **Delta Lake:** Used Delta Lake for target tables to ensure ACID compliance and performance.
6. **Performance Configurations:** Applied Spark optimizations for adaptive query execution and partitioning.

This code is production-ready and adheres to the provided optimization framework.